In [20]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [21]:
bank_df = pd.read_csv("week5-Data/bank-sample.csv")

In [22]:
bank_df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,subscribed
0,31,management,single,tertiary,no,0,yes,no,cellular,15,apr,185,2,-1,0,unknown,no
1,45,entrepreneur,married,tertiary,no,1752,yes,yes,cellular,20,nov,56,2,-1,0,unknown,no
2,46,services,divorced,secondary,no,4329,no,no,cellular,21,nov,534,2,-1,0,unknown,yes
3,35,management,married,tertiary,no,1108,yes,no,cellular,17,nov,52,1,-1,0,unknown,no
4,39,management,married,secondary,no,1410,yes,no,unknown,23,may,55,1,-1,0,unknown,no


In [23]:
bank_df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000,2000.000000
mean,40.868000,1396.779500,15.82200,255.817000,2.655500,38.164000,0.530500
std,10.763527,3063.927783,8.29468,262.768065,3.055076,97.584817,1.810721
min,18.000000,-1854.000000,1.00000,5.000000,1.000000,-1.000000,0.000000
25%,32.000000,79.500000,8.00000,104.000000,1.000000,-1.000000,0.000000
50%,39.000000,463.500000,16.00000,178.000000,2.000000,-1.000000,0.000000
75%,49.000000,1407.750000,21.00000,309.000000,3.000000,-1.000000,0.000000
max,95.000000,36221.000000,31.00000,3881.000000,63.000000,674.000000,37.000000


In [24]:
bank_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   age         2000 non-null   int64 
 1   job         2000 non-null   object
 2   marital     2000 non-null   object
 3   education   2000 non-null   object
 4   default     2000 non-null   object
 5   balance     2000 non-null   int64 
 6   housing     2000 non-null   object
 7   loan        2000 non-null   object
 8   contact     2000 non-null   object
 9   day         2000 non-null   int64 
 10  month       2000 non-null   object
 11  duration    2000 non-null   int64 
 12  campaign    2000 non-null   int64 
 13  pdays       2000 non-null   int64 
 14  previous    2000 non-null   int64 
 15  poutcome    2000 non-null   object
 16  subscribed  2000 non-null   object
dtypes: int64(7), object(10)
memory usage: 265.8+ KB


In [25]:
#selecting categorical columns

categorical_cols = bank_df.select_dtypes(include=['object']).columns

In [26]:
bank_df_encoded = bank_df.copy()

In [27]:
binary_cols = ['default', 'housing', 'loan', 'subscribed']
le = LabelEncoder()

for col in binary_cols:
    print(f"Encoding {col}: {bank_df_encoded[col].unique()}")
    bank_df_encoded[col] = le.fit_transform(bank_df_encoded[col])
    print(f"  -> Encoded values: {bank_df_encoded[col].unique()}")


Encoding default: ['no' 'yes']
  -> Encoded values: [0 1]
Encoding housing: ['yes' 'no']
  -> Encoded values: [1 0]
Encoding loan: ['no' 'yes']
  -> Encoded values: [0 1]
Encoding subscribed: ['no' 'yes']
  -> Encoded values: [0 1]


In [28]:
multi_class_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']

print(f"Original shape: {bank_df_encoded.shape}")

# Use pd.get_dummies for one-hot encoding (drop_first=True to avoid multicollinearity)
bank_df_encoded = pd.get_dummies(bank_df_encoded, columns=multi_class_cols, drop_first=True)

print(f"After one-hot encoding shape: {bank_df_encoded.shape}")
print(f"New columns created: {bank_df_encoded.shape[1] - bank_df.shape[1] + len(multi_class_cols)}")
print("\n")

Original shape: (2000, 17)
After one-hot encoding shape: (2000, 43)
New columns created: 32




In [29]:
print("Encoded DataFrame - First 5 Rows:")
print("=" * 50)
print(bank_df_encoded.head())
print("\n")

print("Encoded DataFrame Info:")
print("=" * 50)
bank_df_encoded.info()
print("\n")

print("Column names after encoding:")
print("=" * 50)
print(list(bank_df_encoded.columns))

Encoded DataFrame - First 5 Rows:
   age  default  balance  housing  loan  day  duration  campaign  pdays  \
0   31        0        0        1     0   15       185         2     -1   
1   45        0     1752        1     1   20        56         2     -1   
2   46        0     4329        0     0   21       534         2     -1   
3   35        0     1108        1     0   17        52         1     -1   
4   39        0     1410        1     0   23        55         1     -1   

   previous  ...  month_jul  month_jun  month_mar  month_may  month_nov  \
0         0  ...      False      False      False      False      False   
1         0  ...      False      False      False      False       True   
2         0  ...      False      False      False      False       True   
3         0  ...      False      False      False      False       True   
4         0  ...      False      False      False       True      False   

   month_oct  month_sep  poutcome_other  poutcome_success  poutc

In [30]:
X = bank_df_encoded.drop('subscribed', axis=1)
y = bank_df_encoded['subscribed']

In [31]:
from sklearn.model_selection import train_test_split
# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y,
train_size=.7, random_state=3011)
# Check the splits are correct
print(f"Train size: {round(len(X_train) / len(X) * 100)}%")
print(f"Test size: {round(len(X_test) / len(X) * 100)}%")

Train size: 70%
Test size: 30%


In [32]:
# Identify numerical columns (excluding encoded categorical variables)
numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
print("Numerical columns to scale:", numerical_cols)

Numerical columns to scale: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


In [33]:
# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler ONLY on X_train and transform X_train
X_train_scaled = X_train.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# Transform X_test using the SAME scaler (do NOT fit again)
X_test_scaled = X_test.copy()
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("Scaling complete!")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

Scaling complete!
X_train_scaled shape: (1400, 42)
X_test_scaled shape: (600, 42)


In [35]:
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)

# Fit the model on the scaled training data
log_reg.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


In [36]:
# Make predictions on both train and test sets
y_train_pred = log_reg.predict(X_train_scaled)
y_test_pred = log_reg.predict(X_test_scaled)

print(f"Training predictions shape: {y_train_pred.shape}")
print(f"Test predictions shape: {y_test_pred.shape}")

Training predictions shape: (1400,)
Test predictions shape: (600,)


In [37]:
# Import metrics for evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

Training Accuracy: 0.9129 (91.29%)
Test Accuracy: 0.8883 (88.83%)


In [38]:
# Import SVM classifier
from sklearn.svm import SVC

In [39]:
# Initialize the SVM model with RBF kernel
svm_model = SVC(kernel='rbf', random_state=42)

# Fit the model on the scaled training data
svm_model.fit(X_train_scaled, y_train)

print("SVM model trained successfully!")

SVM model trained successfully!


In [40]:
# Make predictions on both train and test sets
y_train_pred_svm = svm_model.predict(X_train_scaled)
y_test_pred_svm = svm_model.predict(X_test_scaled)

print(f"Training predictions shape: {y_train_pred_svm.shape}")
print(f"Test predictions shape: {y_test_pred_svm.shape}")

Training predictions shape: (1400,)
Test predictions shape: (600,)


In [41]:
# Calculate accuracy
train_accuracy_svm = accuracy_score(y_train, y_train_pred_svm)
test_accuracy_svm = accuracy_score(y_test, y_test_pred_svm)

print(f"Training Accuracy: {train_accuracy_svm:.4f} ({train_accuracy_svm*100:.2f}%)")
print(f"Test Accuracy: {test_accuracy_svm:.4f} ({test_accuracy_svm*100:.2f}%)")

Training Accuracy: 0.9207 (92.07%)
Test Accuracy: 0.8833 (88.33%)


In [42]:
# Print detailed classification report for test set
print("Classification Report (Test Set) - SVM:")
print("="*60)
print(classification_report(y_test, y_test_pred_svm))

Classification Report (Test Set) - SVM:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94       529
           1       0.57      0.06      0.10        71

    accuracy                           0.88       600
   macro avg       0.73      0.53      0.52       600
weighted avg       0.85      0.88      0.84       600



In [43]:
# Display confusion matrix
print("Confusion Matrix (Test Set) - SVM:")
print("="*60)
cm_svm = confusion_matrix(y_test, y_test_pred_svm)
print(cm_svm)
print("\nInterpretation:")
print(f"True Negatives: {cm_svm[0,0]}")
print(f"False Positives: {cm_svm[0,1]}")
print(f"False Negatives: {cm_svm[1,0]}")
print(f"True Positives: {cm_svm[1,1]}")

Confusion Matrix (Test Set) - SVM:
[[526   3]
 [ 67   4]]

Interpretation:
True Negatives: 526
False Positives: 3
False Negatives: 67
True Positives: 4
